<a href="https://colab.research.google.com/github/Hanu0745/ai-mentor-portfolio/blob/main/Day_2Resumeextractor.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q google-genai pydantic
import os, getpass
if 'GEMINI_API_KEY' not in os.environ:
    os.environ['GEMINI_API_KEY'] = getpass.getpass('Gemini API key: ')

Gemini API key: ··········


In [ ]:
from pydantic import BaseModel
from typing import List, Optional

class Education(BaseModel):
    degree: str
    institution: str
    year: int

class Resume(BaseModel):
    name: str
    email: str
    phone: Optional[str] = None
    education: List[Education]
    skills: List[str]
    projects: List[str] = []
    experience_years: float

In [ ]:
from google import genai
from pydantic import ValidationError

client = genai.Client(api_key=os.environ['GEMINI_API_KEY'])

def extract_resume(raw_text: str, max_retries: int = 1) -> Resume:
    for attempt in range(max_retries + 1):
        try:
            resp = client.models.generate_content(
                model='gemini-2.5-flash',
                contents=f'Extract a Resume JSON from this text. Return ONLY JSON, no markdown.\n\n{raw_text}',
                config={'response_mime_type': 'application/json',
                        'response_schema': Resume.model_json_schema()},
            )
            return Resume.model_validate_json(resp.text)
        except ValidationError as e:
            if attempt == max_retries:
                raise
            # Retry once: ask Gemini to fix the JSON
            fix_prompt = f'Fix this JSON to match schema. Errors: {e}. Original: {resp.text}'
            resp = client.models.generate_content(
                model='gemini-2.5-flash', contents=fix_prompt,
                config={'response_mime_type': 'application/json',
                        'response_schema': Resume.model_json_schema()},
            )
            return Resume.model_validate_json(resp.text)

In [ ]:
# Load sample résumés
with open('/content/spring_boot_mcq_test.txt') as f:
    resumes = [r.strip() for r in f.read().split('---') if r.strip()]

results = []
for i, r in enumerate(resumes[:3]):
    try:
        parsed = extract_resume(r)
        results.append(parsed)
        print(f'Résumé {i+1}: {parsed.name} — {len(parsed.skills)} skills')
    except Exception as e:
        print(f'Résumé {i+1}: FAILED — {e}')

print('\n', results[0].model_dump_json(indent=2) if results else 'no results')

Résumé 1: Ravi Kumar — 6 skills
Résumé 2: Sneha Reddy — 6 skills
Résumé 3: Arun Pillai — 8 skills

 {
  "name": "Ravi Kumar",
  "email": "ravi.kumar@example.com",
  "phone": "+91-9876543210",
  "education": [
    {
      "degree": "B.Tech Computer Science",
      "institution": "Aditya University",
      "year": 2026
    },
    {
      "degree": "Intermediate",
      "institution": "Narayana Junior College",
      "year": 2022
    }
  ],
  "skills": [
    "Python",
    "Java",
    "SQL",
    "Git",
    "Linux",
    "REST APIs"
  ],
  "projects": [
    "Built a Flask REST API for college placement portal (3-week internship at startup)",
    "Solved 250+ DSA problems on LeetCode (Top 5% in college)",
    "Final-year project: ML model for crop yield prediction (Random Forest, 87% accuracy)"
  ],
  "experience_years": 0.25
}


## Extracting MCQs into JSON Format

To convert your MCQs from a text file into a structured JSON format, we will:

1.  Define a Pydantic model to represent the structure of a single Multiple Choice Question (MCQ).
2.  Provide the path to your MCQ text file.
3.  Read the content of the MCQ text file.
4.  Use the Gemini API to extract the MCQs and format them as JSON, leveraging the Pydantic schema for validation and structure.

In [ ]:
# 1. Define Pydantic Models for MCQs

from pydantic import BaseModel, Field
from typing import List

class MCQOption(BaseModel):
    label: str  # e.g., 'A', 'B', 'C', 'D'
    text: str   # e.g., 'Option A text'

class MCQ(BaseModel):
    question: str
    options: List[MCQOption]
    correct_answer_label: str = Field(description="The label (e.g., 'A', 'B') of the correct option")

class Quiz(BaseModel):
    mcqs: List[MCQ]

Please replace `'/content/your_mcqs.txt'` with the actual path to your MCQ text file.

In [ ]:
# 2. & 3. Specify and read the MCQ text file

# !!! IMPORTANT: Replace 'mcqs.txt' with the actual name of your MCQ file !!!
mcq_file_path = '/content/spring_boot_mcq_test.txt' # Assuming you've uploaded a file named 'mcqs.txt'

try:
    with open(mcq_file_path, 'r') as f:
        mcq_raw_text = f.read()
    print(f"Successfully loaded MCQs from {mcq_file_path}. Content preview:\n{mcq_raw_text[:500]}...")
except FileNotFoundError:
    print(f"Error: The file '{mcq_file_path}' was not found. Please ensure the file is uploaded and the path is correct.")
    mcq_raw_text = None # Set to None to prevent further processing if file not found

Successfully loaded MCQs from /content/spring_boot_mcq_test.txt. Content preview:
Spring Boot MCQ Test

Name: _______________________
Date: _______________________

Instructions:
- Choose the correct answer for each question.
- Each question carries 1 mark.

--------------------------------------------------

1. What is the main purpose of Spring Boot?

A. To replace Java completely
B. To simplify Spring application development
C. To create only desktop applications
D. To manage operating systems

Answer: _______

-----------------------------------------...


In [ ]:
# 4. Parse and extract MCQs using Gemini API

def extract_quiz(raw_text: str, max_retries: int = 1) -> Quiz:
    if not raw_text:
        return Quiz(mcqs=[])

    for attempt in range(max_retries + 1):
        try:
            resp = client.models.generate_content(
                model='gemini-2.5-flash',
                contents=f'Extract a Quiz JSON (list of MCQs) from this text. Return ONLY JSON, no markdown.\n\n{raw_text}',
                config={'response_mime_type': 'application/json',
                        'response_schema': Quiz.model_json_schema()},
            )
            return Quiz.model_validate_json(resp.text)
        except ValidationError as e:
            print(f"Validation error on attempt {attempt+1}: {e}")
            if attempt == max_retries:
                raise
            # Retry once: ask Gemini to fix the JSON
            fix_prompt = f'Fix this JSON to match schema. Errors: {e}. Original: {resp.text}'
            resp = client.models.generate_content(
                model='gemini-2.5-flash', contents=fix_prompt,
                config={'response_mime_type': 'application/json',
                        'response_schema': Quiz.model_json_schema()},
            )
            return Quiz.model_validate_json(resp.text)
        except Exception as e:
            print(f"An unexpected error occurred: {e}")
            raise

if mcq_raw_text:
    try:
        parsed_quiz = extract_quiz(mcq_raw_text)
        print(f"Successfully extracted {len(parsed_quiz.mcqs)} MCQs.")
        print("\n--- Generated JSON ---\n")
        print(parsed_quiz.model_dump_json(indent=2))
    except Exception as e:
        print(f"Failed to extract quiz: {e}")
else:
    print("No MCQ raw text to process due to previous file loading error.")

Successfully extracted 10 MCQs.

--- Generated JSON ---

{
  "mcqs": [
    {
      "question": "What is the main purpose of Spring Boot?",
      "options": [
        {
          "label": "A",
          "text": "To replace Java completely"
        },
        {
          "label": "B",
          "text": "To simplify Spring application development"
        },
        {
          "label": "C",
          "text": "To create only desktop applications"
        },
        {
          "label": "D",
          "text": "To manage operating systems"
        }
      ],
      "correct_answer_label": "B"
    },
    {
      "question": "Which annotation is used to mark the main class in a Spring Boot application?",
      "options": [
        {
          "label": "A",
          "text": "@SpringApp"
        },
        {
          "label": "B",
          "text": "@EnableBoot"
        },
        {
          "label": "C",
          "text": "@SpringBootApplication"
        },
        {
          "label": "D",
